In [1]:
# normalize column names to lowercase_with_underscores
df.columns = (
    df.columns
    .str.strip()                                    # remove stray leading/trailing spaces
    .str.lower()                                    # Speed_MPH → speed_mph
    .str.replace(r"\W+", "_", regex=True)           # any non-word character → underscore
)

df = df.drop_duplicates()                           # remove rows identical in every column

# duplicate timestamps that DISAGREE — collapse to the median
df = df.groupby("timestamp", as_index=False).agg(   # one row per timestamp
    speed_mph=("speed_mph", "median"),              # median of that timestamp's readings
    segment_id=("segment_id", "first"),             # keep the first (they're all the same)
)

df = df.sort_values("timestamp").reset_index(drop=True)  # chronological; drop=True discards old row numbers

NameError: name 'df' is not defined

In [ ]:
SENSOR_ERROR_CODES = [255, -1]                      # vendor's "no reading" placeholders
df.loc[df["speed_mph"].isin(SENSOR_ERROR_CODES), "speed_mph"] = np.nan  # .loc[rows, col] = value

# zeros: sustained 0 mph on a highway is usually a stuck sensor,
# but a single 0 during a jam can be real. Look before deciding.
print(df.loc[df["speed_mph"] == 0, "timestamp"].dt.hour.value_counts())  # what hours do zeros occur?

In [ ]:
df = df.set_index("timestamp")                      # timestamp becomes the row label, not a column
full = pd.date_range(                               # the complete grid that SHOULD exist
    df.index.min(), df.index.max(), freq="5min"
)
df = df.reindex(full)                               # insert blank rows wherever a timestamp is missing
df.index.name = "timestamp"                         # reindex drops the name; restore it

completeness = df["speed_mph"].notna().mean()       # fraction of rows with a real value
print(f"data completeness: {completeness:.1%}")     # .1% formats 0.9423 as "94.2%"

In [ ]:
df["speed_filled"] = df["speed_mph"].interpolate(
    limit=3,                                        # fill runs of at most 3 consecutive NaNs (15 min)
    limit_area="inside",                            # never extrapolate past the first/last real value
)